In [2]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer

In [3]:
data = load_breast_cancer()

# Inputs
X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

# Outputs
y = pd.Series(data.target)

In [4]:
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
X.shape

(569, 30)

In [6]:
y.value_counts()

1    357
0    212
Name: count, dtype: int64

In [ ]:
# Train/Test Split
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, # 80% training, 20% testing
    random_state=42,
    stratify=y # keep both sets roughly the same ratio
)

In [8]:
print(X_train.shape) # (rows, columns)
print(X_test.shape)

print(y_train.value_counts(normalize=True)) # normalize=True changes value_counts from counts to
print(y_test.value_counts(normalize=True))

(455, 30)
(114, 30)
1    0.626374
0    0.373626
Name: proportion, dtype: float64
1    0.631579
0    0.368421
Name: proportion, dtype: float64


# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Creates logistic regression model
model = LogisticRegression(
    max_iter=1000
)

# Trains the logistic regression using training data
model.fit(
    X_train,
    y_train
)

# Use trained model to make predictions on unseen data
predictions = model.predict(X_test)

# Evaluate performance by comparing y_test(actual) and predictions
accuracy_score(y_test, predictions)

/home/sinqiao/Missing-Data-ML/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


The model reached iteration 1000 but still had not reached the optimiser's convergence criterion.

The most common reason is features being on very different scales.

The optimiser can have a harder time finding the optimum when one feature is measured in tens and another in hundreds of thousands.

Recommended solution: scale the features

### Add scaling

Scaling transformed the features so they have
mean ≈ 0
standard deviation ≈ 1

Example:

Before scaling:

radius_mean       14.1
area_mean        654.9
smoothness       0.09

After scaling:

radius_mean       0.35
area_mean         0.52
smoothness       -0.21

The optimizer usually converges much faster.

In [31]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000)
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print(accuracy)

0.9824561403508771


# Random Forest

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create the Random Forest model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Make predictions on test data
predictions = model.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, predictions)

print(accuracy)

0.956140350877193


## Evalute both

In [40]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Logistic Regression
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000)
)

logistic_model.fit(X_train, y_train)


# Random Forest
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest_model.fit(X_train, y_train)


# Evaluation function
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    return accuracy

logistic_accuracy = evaluate_model(logistic_model, X_test, y_test)
random_forest_accuracy = evaluate_model(random_forest_model, X_test, y_test)

# Evaluate both models
print("Logistic Regression:", logistic_accuracy)
print("Random Forest:", random_forest_accuracy)

Logistic Regression: 0.9824561403508771
Random Forest: 0.956140350877193


## Create pandas DataFrame

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Logistic Regression
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

logistic_accuracy = accuracy_score(y_test, logistic_predictions)
logistic_f1 = f1_score(y_test, logistic_predictions)
logistic_auc = roc_auc_score(y_test, logistic_probabilities)

# Random Forest
random_forest_predictions = random_forest_model.predict(X_test)
random_forest_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

random_forest_accuracy = accuracy_score(y_test, random_forest_predictions)
random_forest_f1 = f1_score(y_test, random_forest_predictions)
random_forest_auc = roc_auc_score(y_test, random_forest_probabilities)

# Create results DataFrame
results_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [logistic_accuracy, random_forest_accuracy],
    "F1": [logistic_f1, random_forest_f1],
    "AUC": [logistic_auc, random_forest_auc]
})

print(results_df)

                 Model  Accuracy        F1       AUC
0  Logistic Regression  0.982456  0.986111  0.995370
1        Random Forest  0.956140  0.965517  0.993717


In [45]:
results_df.to_csv("../results/02results.csv", index=False)

# Discussion

## Observations

- Both Logistic Regression and Random Forest achieved good performance on the Breast Cancer Wisconsin dataset.
- Logistic Regression provides a simple and interpretable baseline for comparison.
- Random Forest achieved comparable performance and can capture nonlinear relationships between features.
- Logistic Regression performed slightly better in accuracy and F1, achieving an accuracy of 0.97 and an F1 score of 0.97, compared with 0.96 for Random Forest on both metrics.
- Both models achieved a high AUC of 0.99, indicating strong ability to distinguish between the two classes.
- The evaluation metrics (Accuracy, F1, and AUC) provide different perspectives on model performance.
- These baseline results will serve as a reference when evaluating different missing-data imputation methods in later experiments.